# DAY2 실습 01 — 분석 단위와 집단별 기술통계

In [ ]:
%pip install -q koreanize-matplotlib

import numpy as np
import pandas as pd
import koreanize_matplotlib

In [ ]:
import os

from google.colab import drive

drive.mount("/content/drive")
data_folder = "/content/drive/MyDrive/HD-AI/data"
csv_path = os.path.join(data_folder, "superstore.csv")

df = pd.read_csv(csv_path)

data = df.copy()

Mounted at /content/drive


## 데이터와 질문

superstore.csv는 온라인 매장의 주문 기록이다. 한 행은 **주문 안의 상품 한 줄(상품행)**이다. 한 주문에 상품이 여러 개 들어 있으면 상품행도 여러 줄이 되므로 행 수와 주문 수는 다르다.

| 열 | 뜻 |
|---|---|
| order_id | 주문번호. 같은 주문의 상품행은 같은 번호를 가진다 |
| product_id | 상품번호 |
| customer_id | 고객번호 |
| region | 지역 : Central · East · South · West |
| segment | 고객군 : Consumer · Corporate · Home Office |
| category | 상품 분류 : Furniture · Office Supplies · Technology |
| sales | 상품행 매출 |
| quantity | 상품행 판매수량 |
| profit | 상품행 이익 (손해가 나면 음수) |

매출과 이익은 CSV의 금액 단위를 그대로 사용한다.

집단별로 비교하기 전에 아래 네 가지를 먼저 정한다.

| 구분 | 확인할 질문 | 예 : 지역별 주문당 평균매출 |
|---|---|---|
| 분석 단위 | 무엇을 1건으로 세는가 | 주문 1건 |
| 집단 기준 | 무엇으로 나누어 비교하는가 | region |
| 분석 변수 | 무엇을 계산하는가 | 주문매출 |
| 요약 방법 | 어떤 통계량으로 요약하는가 | 평균 |

### 집단별 결과를 읽을 때 확인할 것

같은 데이터라도 분석 단위, 집단 기준, 집계 방법에 따라 결과의 의미가 달라진다.

| 확인할 것 | 주의할 점 |
|---|---|
| 한 행의 의미 | 상품행 수를 주문 수나 고객 수로 해석하지 않는다 |
| 합계 | 판매수량의 합과 주문 건수는 같은 의미가 아니다 |
| 평균·중앙값 | 상품행당 평균과 주문당 평균을 구분한다 |
| 가중평균 | 평균을 구하려는 분석 단위에 맞는 가중치를 사용한다 |
| 총량과 평균 | 총량이 큰 집단이 평균도 가장 높다고 단정하지 않는다 |

## 1. 행 수와 고유 대상 수

같은 데이터라도 무엇을 세느냐에 따라 숫자가 달라진다. 행 수, 유효관측수, 고유값 수, 수량 합계를 구분한다.

| 계산 | 세는 대상 | 결측값 |
|---|---|---|
| `len(데이터)` | 전체 행 수 | 포함해서 센다 |
| `데이터["열"].count()` | 그 열에서 값이 있는 행 수 (유효관측수) | 빼고 센다 |
| `데이터["열"].nunique()` | 그 열에서 서로 다른 값의 수 (고유값 수) | 빼고 센다 |
| `데이터["수량열"].sum()` | 수량의 합계 | 빼고 더한다 |


공식 문서 : [count](https://pandas.pydata.org/docs/reference/api/pandas.Series.count.html) · [nunique](https://pandas.pydata.org/docs/reference/api/pandas.Series.nunique.html) · [sum](https://pandas.pydata.org/docs/reference/api/pandas.Series.sum.html)

In [ ]:
# [예제] 작은 주문표에서 행 수 · 유효관측수 · 주문 수 · 판매수량 구분
# 주문 3건에 상품행 5줄이 있고, 매출 하나가 비어 있다(NaN)

sample = pd.DataFrame({
    "order_id": ["O1", "O1", "O2", "O3", "O3"],
    "product_id": ["P1", "P2", "P1", "P3", "P2"],
    "sales": [200, 150, np.nan, 500, 80],
    "quantity": [2, 1, 3, 1, 4],
})
display(sample)

print("상품행 수:", len(sample))                       # 결측과 상관없이 행을 모두 센다
print("매출 유효관측수:", sample["sales"].count())      # 매출이 비어 있는 행은 빼고 센다
print("고유 주문 수:", sample["order_id"].nunique())   # 같은 주문번호는 한 번만 센다
print("판매수량:", sample["quantity"].sum())           # quantity 값을 모두 더한다

,order_id,product_id,sales,quantity
0,O1,P1,200.0,2
1,O1,P2,150.0,1
2,O2,P1,NaN,3
3,O3,P3,500.0,1
4,O3,P2,80.0,4


상품행 수: 5
매출 유효관측수: 4
고유 주문 수: 3
판매수량: 11


In [ ]:
# [실습]
# data 에서 상품행 수, 고유 주문·상품·고객 수, 판매수량은 각각 얼마인가?



## 2. 집단별로 나누어 요약하기

`data.groupby("region")["sales"].sum()`은 region별로 행을 나눈 뒤, 각 집단의 sales를 더한다.

| 부분 | 의미 | 예 |
|---|---|---|
| `데이터` | 분석할 DataFrame | `data` |
| `groupby("그룹기준열")` | 데이터를 나누는 기준 | `groupby("region")` |
| `["분석변수"]` | 각 집단에서 계산할 열 | `["sales"]` |
| `집계함수()` | 합계·평균 등 요약 방법 | `sum()`, `mean()` |

집단마다 개수를 셀 때도 무엇을 세는지 구분한다.

| 계산 | 세는 대상 |
|---|---|
| `groupby("기준열").size()` | 집단별 전체 행 수 (결측 포함) |
| `groupby("기준열")["열"].count()` | 집단별 유효관측수 (결측 제외) |
| `groupby("기준열")["열"].nunique()` | 집단별 고유값 수 |

공식 문서 : [groupby](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.groupby.html) · [그룹별 집계](https://pandas.pydata.org/docs/user_guide/groupby.html#aggregation)

In [ ]:
# [예제] 지역별 상품행 수 · 매출 유효관측수 · 고유 주문 수
# 문제: 지역별 전체 상품행 수, 매출 유효관측수, 중복을 제외한 고유 주문 수를 계산한다.


# 1. size() : 지역별 전체 행 수
#    sales에 결측값이 있어도 행 개수에 포함한다.
row_n = data.groupby("region").size()


# 2. count() : 지역별 sales의 유효값 개수
#    sales가 NaN인 행은 제외하고 센다.
valid_n = data.groupby("region")["sales"].count()


# 3. nunique() : 지역별 서로 다른 주문번호 개수
#    같은 order_id가 여러 번 나와도 1개로 센다.
order_n = data.groupby("region")["order_id"].nunique()


# 4. 결과를 하나의 DataFrame으로 정리
counts = pd.DataFrame({
    "상품행 수": row_n,
    "매출 유효관측수": valid_n,
    "고유 주문 수": order_n
})

display(counts)

,상품행 수,매출 유효관측수,고유 주문 수
region,,,
Central,2335,2335,1179
East,2986,2986,1475
South,1620,1620,822
West,3253,3253,1635


In [ ]:
# [실습]
# 상황 : 지역별로 얼마나 많이 팔았는지 비교
# 지역별 총매출과 주문 수는 얼마인가?



In [ ]:
# [실습]
# 상황 : 지역마다 고객이 한 번 주문할 때 상품을 몇 개씩 사는지 비교
# 지역별 판매수량 합계와 주문 수를 구하고, 주문 1건당 평균 판매수량을 구하시오. (소수점 둘째 자리)



### 직접 해보기

카테고리(category)별 총매출, 판매수량, 주문 수를 구하고, 카테고리별 주문 수를 모두 더한 값이 전체 주문 수와 같은지 확인한다.

In [ ]:
# [실습]
# 상황 : 상품 종류별 매출 규모를 비교한다.
# 카테고리(category)별 총매출, 판매수량, 주문 수를 구하시오.
# 카테고리별 주문 수를 모두 더하면 전체 주문 수와 같은가? 다르다면 이유는?



## 3. 주문 단위로 묶어 집단별로 요약하기

하나의 주문번호(`order_id`)에는 여러 상품이 들어 있을 수 있다.

| order_id | region | segment | 상품 | sales |
|---|---|---|---|---:|
| O001 | 서울 | 일반 | 노트북 | 100 |
| O001 | 서울 | 일반 | 마우스 | 200 |
| O002 | 부산 | 기업 | 모니터 | 300 |
| O003 | 서울 | 일반 | 키보드 | 150 |

O001은 **상품행이 2개이지만 주문은 1건**이다. 주문당 매출을 구하려면 같은 주문번호의 상품행 매출을 더한다.

| order_id | region | segment | 주문매출 |
|---|---|---|---:|
| O001 | 서울 | 일반 | 300 |
| O002 | 부산 | 기업 | 300 |
| O003 | 서울 | 일반 | 150 |

상품행 4개가 주문 3건으로 집계된다. 이 표에서 주문매출의 평균을 구하면 **주문 1건당 평균 매출**이 된다.

### 여러 열을 한 번에 집계하는 `agg()`

`agg(새열이름=("원래열", "집계방법"))` 형식으로 결과 열 이름과 계산 방법을 함께 정한다.

| 코드 | 의미 |
|---|---|
| `groupby("order_id")` | 같은 주문번호의 상품행을 묶는다 |
| `as_index=False` | 주문번호를 일반 열로 남긴다 |
| `region=("region", "first")` | 주문의 지역을 가져온다 |
| `주문매출=("sales", "sum")` | 상품행 매출을 더해 주문매출을 구한다 |

- `first`는 각 열에서 처음 나오는 결측이 아닌 값을 가져온다. 주문의 지역·고객군을 대표값으로 쓰려면 같은 주문 안에서 값이 일관되어야 한다.
- `sum`은 집단에 속한 값을 더한다. 상품행별 매출을 더하면 주문매출이 된다.

같은 주문에 지역이나 고객군이 여러 개 기록되어 있으면 `first`를 사용할 때 정보가 빠질 수 있다. 그래서 아래 예제에서 `nunique()`로 주문별 지역·고객군이 하나씩인지 먼저 확인한 뒤 집계한다. `nunique()`는 결측치를 빼고 세므로 결측 여부도 함께 확인한다.

공식 문서 : [이름 붙여 집계하기](https://pandas.pydata.org/docs/user_guide/groupby.html#named-aggregation)

In [ ]:
# [예제] 주문번호별 지역과 고객군의 일관성 확인
# 문제: 같은 주문번호에 지역이나 고객군이 둘 이상 기록되어 있는가?

# 1. 주문번호별로 그룹화
order_keys = data.groupby("order_id")[["region", "segment"]].nunique()

# 2. 지역과 고객군의 고유값 개수 중 최댓값 확인
display(order_keys.max())

,0
region,1
segment,1


In [ ]:
# [실습]
# 상황 : 주문 1건당 매출을 확인한다.
# order_id를 기준으로 상품행을 묶어 주문별 매출과 이익을 계산하시오.
# region과 segment는 첫 번째 값을 가져오시오.
# 집계 후 주문 수와 고유 주문 수를 비교하시오.
# 주문 단위 표는 orders 라는 이름에 담는다


In [ ]:
# [실습]
# 상황 : 지역별로 주문 1건의 매출 수준(평균·중앙값)과 퍼짐(표준편차·IQR)을 한 표로 비교한다.
# 지역별 주문 수, 총매출, 평균, 중앙값, 표준편차, Q1·Q3·IQR은 얼마인가?
# 결과 표는 summary 라는 이름에 담고, 열 이름은 n · 합계 · 평균 · 중앙값 · 표준편차 · Q1 · Q3 · IQR 로 한다



### 직접 해보기


In [ ]:
# [실습]
# 상황 : 어느 고객군이 한 번 주문할 때 더 큰 금액을 주문하는지 비교한다.
# 고객군(segment)별 주문 수와 주문당 평균·중앙값을 계산하고, 평균과 중앙값을 함께 비교하시오.
# 계산 결과를 근거로 관찰 문장 한 개를 작성하시오.



### 직접 해보기

지역별 주문매출의 변동계수(CV, %)를 구하고, 표준편차 순위와 비교한다. CV = 표준편차 ÷ 평균 × 100이다.

In [ ]:
# [실습]
# 상황 : 지역마다 평균 주문매출이 다르다. 평균 규모가 다른 지역의 퍼짐을 공정하게 비교한다.
# summary 를 이용해 지역별 주문매출의 CV(%)를 구하고, 표준편차와 나란히 놓아 비교하시오. (소수점 둘째 자리)



## 4. 평균의 평균과 가중평균

집단별 평균을 합칠 때는 각 집단에 같은 비중을 줄지, 집단 크기를 반영할지 먼저 정한다. 집단 크기를 반영해 합친 평균을 **가중평균**(Weighted Mean)이라고 한다.

| 계산 | 방법 | 비중 |
|---|---|---|
| 평균의 평균 | 집단 평균의 합 ÷ 집단 수 | 집단마다 같은 비중 |
| 가중평균 | (집단 평균 × 집단 크기)의 합 ÷ 전체 건수 | 관측 한 건마다 같은 비중 |



### 확인할 점

가중평균이 전체 평균과 같아지려면 대상 전체가 집단에 **결측없이, 중복 없이** 들어 있어야 한다.



**평균을 계산하기 전에 상품행, 주문, 상품 1개, 고객 중 무엇을 한 건으로 볼지 먼저 정한다.**

공식 문서 : [mean](https://pandas.pydata.org/docs/reference/api/pandas.Series.mean.html) · [sum](https://pandas.pydata.org/docs/reference/api/pandas.Series.sum.html) · [count](https://pandas.pydata.org/docs/reference/api/pandas.Series.count.html)

In [ ]:
# [예제] 평균의 평균과 가중평균
# 서울 주문 2건(주문당 평균 100), 부산 주문 8건(주문당 평균 200)

ex = pd.DataFrame({"주문수": [2, 8], "평균매출": [100, 200]}, index=["서울", "부산"])
display(ex)

# 평균의 평균 : 두 지역에 같은 비중을 준다
print("평균의 평균:", ex["평균매출"].mean())

# 가중평균 : (지역별 평균 × 주문 수)를 모두 더한 뒤 전체 주문 수로 나눈다
print("가중평균:", (ex["평균매출"] * ex["주문수"]).sum() / ex["주문수"].sum())

,주문수,평균매출
서울,2,100
부산,8,200


평균의 평균: 150.0
가중평균: 180.0


In [ ]:
# [실습]
# 상황 : 본사 보고서에 지역별 '주문당 평균매출'과 '주문 수'만 있고, 주문 원자료는 없다고 하자.
#        이 요약표만으로 회사 전체의 주문 1건당 평균매출을 구해야 한다.
# 지역별 주문당 평균매출의 단순 평균과 전체 주문당 평균은 같은가?



### 직접 해보기


In [ ]:
# [실습]
# 상황 : 상품 1개를 평균 얼마에 팔았는지(평균 판매단가)를 보고해야 한다.
#        데이터에는 상품행마다 매출과 판매수량만 있다.
# 판매한 상품 1개당 평균단가는 상품행 단가의 단순 평균과 같은가?
# 단, quantity 에 0 이 없고 sales·quantity 에 결측값이 없다고 본다



## 5. 정리

| 확인할 것 | 함수·계산 | 답하는 질문 |
|---|---|---|
| 전체 행 수 | `len()` | 상품행이 몇 줄인가 |
| 유효관측수 | `count()` | 값이 있는 행이 몇 개인가 |
| 고유 대상 수 | `nunique()` | 주문·상품·고객이 몇 개인가 |
| 판매수량 | `["quantity"].sum()` | 몇 개가 팔렸는가 |
| 집단별 요약 | `groupby("기준열")["열"].집계함수()` | 집단마다 합계·평균은 얼마인가 |
| 여러 통계량을 한 표로 | `agg(이름="함수")` | 집단별 n·평균·중앙값·표준편차를 한 번에 |
| 가중평균 | `(평균 × n).sum() / n.sum()` | 집단 크기를 반영한 전체 평균은 얼마인가 |

순서
1) 한 행이 무엇인지 확인하고, 질문에서 1건이 무엇인지 정한다 (분석 단위)
2) 필요하면 분석 단위로 묶는다 (상품행 → 주문)
3) 집단 기준으로 나누고 집단 크기 n을 먼저 본다
4) 평균과 중앙값, 표준편차와 IQR을 함께 본다. 평균 규모가 다르면 CV도 본다
5) 집단 평균으로 전체 평균을 구할 때는 집단 크기를 가중치로 쓴다

공식 문서 : [groupby](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.groupby.html) · [이름 붙여 집계하기 (agg)](https://pandas.pydata.org/docs/user_guide/groupby.html#named-aggregation) · [count](https://pandas.pydata.org/docs/reference/api/pandas.Series.count.html) · [nunique](https://pandas.pydata.org/docs/reference/api/pandas.Series.nunique.html)

## 6. 종합 실습 : 지역별 주문당 평균이익

3절에서 만든 주문 단위 표 `orders`를 사용한다. 주문이익은 손해가 난 주문에서 음수가 될 수 있다.

In [ ]:
# 문제 1)
# 상황 : 지역별 수익성을 비교하고, 지역 요약표만으로 회사 전체의 주문당 평균이익을 구해 본다.
# 지역별 주문 수(n)와 주문당 평균이익을 구하시오.



In [ ]:
# 문제 2)
# 지역별 평균이익의 단순 평균과 주문 수 가중평균을 구하시오.



In [ ]:
# 문제 3)
# orders 전체에서 주문당 평균이익을 직접 구하고, 문제 2의 두 값과 비교하시오.

